# **This notebook acts as the data preprocessor of MIMIC-III**

# **Now ICU stays as aggregation template with mostly time series data**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder

## Loading our aggregated data

In [ ]:
client_1 = pd.read_csv('client_1_ICUSTAY_raw_hour.csv', low_memory=False)

In [ ]:
client_1

## Understadig the structure of the dataset

In [ ]:
# Basic structural overview
num_rows, num_columns = client_1.shape
column_names = client_1.columns.tolist()
data_types = client_1.dtypes
sample_rows = client_1.head(10)

# Count unnamed columns
unnamed_columns = [col for col in column_names if "Unnamed" in col]

# Summary results
structure_summary = {
    "Number of rows": num_rows,
    "Number of columns": num_columns,
    "Unnamed columns": unnamed_columns,
    "Data types sample": data_types.head(10).to_dict()
}

structure_summary

In [ ]:
# Classify columns by data type categories
categorical_cols = []
numerical_cols = []
datetime_cols = []

for col in client_1.columns:
    dtype = client_1[col].dtype
    if dtype == 'object':
        # Check if the column looks like a date
        try:
            pd.to_datetime(client_1[col].dropna().iloc[0])
            datetime_cols.append(col)
        except Exception:
            categorical_cols.append(col)
    elif pd.api.types.is_numeric_dtype(dtype):
        numerical_cols.append(col)

# Summary of data types
type_summary = {
    "Categorical columns (sample)": categorical_cols[:10],
    "Numerical columns (sample)": numerical_cols[:10],
    "Datetime columns (sample)": datetime_cols[:10],
    "Count - categorical": len(categorical_cols),
    "Count - numerical": len(numerical_cols),
    "Count - datetime-like": len(datetime_cols)
}

type_summary

## Checking the number of missing values

In [ ]:
missing_chunk = client_1.copy()

# Calculate missing value stats
missing_stats = (
    missing_chunk.isnull().sum()
    .to_frame(name='Missing_Count')
    .assign(Total=missing_chunk.shape[0])
    .assign(Missing_Percent=lambda x: (x['Missing_Count'] / x['Total']) * 100)
    .sort_values(by='Missing_Percent', ascending=False)
)

# Filter only columns with missing values
missing_stats_filtered = missing_stats[missing_stats['Missing_Count'] > 0]

missing_stats_filtered.shape[0] # Number of columns with missing values

In [ ]:
# Display top 20 columns with the highest percentage of missing values
top_missing = missing_stats_filtered.head(20).copy()
top_missing.reset_index(inplace=True)
top_missing.rename(columns={'index': 'Column Name'}, inplace=True)

top_missing

In [ ]:
missing_percent = client_1.isnull().mean().sort_values(ascending=False) * 100
missing_percent

In [ ]:
# Define a threshold for dropping columns based on missing percentage
drop_threshold = 95 # percent
columns_to_drop = missing_stats_filtered[missing_stats_filtered['Missing_Percent'] > drop_threshold].index.tolist()

# Separate remaining columns for potential imputation (not to be dropped)
columns_to_impute = missing_stats_filtered[
    (missing_stats_filtered['Missing_Percent'] <= drop_threshold)
].index.tolist()

# Show the first few columns that would be dropped and imputed
drop_and_impute_summary = {
    "Columns to drop (sample)": columns_to_drop[:10],
    "Drop count": len(columns_to_drop),
    "Columns to impute (sample)": columns_to_impute[:10],
    "Impute count": len(columns_to_impute)
}

drop_and_impute_summary

In [ ]:
missing_percent.head(30).plot(kind='bar', figsize=(12, 6), title="Top 30 features with most missing values")
plt.ylabel('% missing')
plt.tight_layout()
plt.ylim(0,100)
plt.show()

In [ ]:
client_1.drop(columns=columns_to_drop, inplace=True)

### Handling capital letters

In [ ]:
client_1.columns = [col.lower() for col in client_1.columns] # Ensure lowercase
print(client_1.columns.tolist())

In [ ]:
client_1

### Like in MIMIC Extract handle cohorts

In [ ]:
# First we try and compute patient age at the time of ICU admission.
client_1['dob'] = pd.to_datetime(client_1['dob'], errors='coerce')
client_1['intime'] = pd.to_datetime(client_1['intime'], errors='coerce')

years = client_1['intime'].dt.year - client_1['dob'].dt.year

had_birthday = ((client_1['intime'].dt.month > client_1['dob'].dt.month) | ((client_1['intime'].dt.month == client_1['dob'].dt.month) & (client_1['intime'].dt.day >= client_1['dob'].dt.day)))

client_1['age'] = years - (~had_birthday).astype(int)

client_1['age'] = client_1['age'].clip(lower=0, upper=89)
client_1.loc[client_1['dob'].isna() | client_1['intime'].isna() | (client_1['dob'] > client_1['intime']), 'age'] = pd.NA

In [ ]:
adult = client_1[client_1['age'] >= 15].copy()

# We ensure only firs stay is kept, so no patient is counted twice
first_stays = adult.drop_duplicates(subset='subject_id', keep='first')

In [ ]:
first_stays = first_stays.copy()

first_stays['intime']  = pd.to_datetime(first_stays['intime'])
first_stays['outtime'] = pd.to_datetime(first_stays['outtime'])

first_stays['los_hours'] = (
    first_stays['outtime'] - first_stays['intime']
).dt.total_seconds() / 3600

cohort = first_stays[
    (first_stays['los_hours'] >= 12) & # 12 hours
    (first_stays['los_hours'] < 240) # 10 days
]

In [ ]:
# Generated code for summary check just to be sure:
print("All stays:", first_stays.shape[0])
print("Filtered cohort:", cohort.shape[0])
print(cohort['los_hours'].describe())

## Handling categorical variables

In [ ]:
cohort = cohort.copy()

cohort['dod'] = pd.to_datetime(cohort['dod'])
cohort['is_alive'] = (cohort['dod'].isna() | (cohort['dod'] > cohort['outtime'])).astype(int)

In [ ]:
# Also gender:
cohort['gender'] = cohort['gender'].map({'M':0,'F':1})

In [ ]:
cohort.columns.tolist()

In [ ]:
# Now define exactly your static columns
static_cols = ['subject_id','icustay_id','age','gender','los_hours','is_alive']
static = cohort[static_cols].copy()

### Handling date time variables

In [ ]:
# From old preprocessing code:
static['admit_hour'] = cohort['intime'].dt.hour
static['admit_weekday']= cohort['intime'].dt.weekday

In [ ]:
# More
static['is_weekend'] = (static['admit_weekday'] >= 5).astype(int)

# Cyclical (sin/cos) encodings so your model “knows” that 23 to 0h is adjacent:

# hour: 0 to 23 circle
static['hour_sin'] = np.sin(2*np.pi * static['admit_hour']   / 24)
static['hour_cos'] = np.cos(2*np.pi * static['admit_hour']   / 24)

# weekday: 0 (Mon) to 6 (Sun)
static['wday_sin'] = np.sin(2*np.pi * static['admit_weekday']/ 7)
static['wday_cos'] = np.cos(2*np.pi * static['admit_weekday']/ 7)

In [ ]:
static

### Putting it all together

In [ ]:
# Get our "dynamic" columns by just taking the rest of the columns which we did not use for static features
dyn_cols = sorted(set(cohort.columns) - set(static_cols))
X_dyn = cohort[dyn_cols].copy()

In [ ]:
import re

# 1) select only columns named like "<var>_h<hour>"
pat = re.compile(r'^(?P<var>.+)_h(?P<hour>\d+)$')
valid = [c for c in X_dyn.columns if pat.match(c)]

hours = sorted({int(pat.match(c).group('hour')) for c in valid})
vars_ = sorted({pat.match(c).group('var') for c in valid})

complete_vars = [
    v for v in vars_
    if all(f"{v}_h{h}" in X_dyn.columns for h in hours)
]

ordered = [
    f"{v}_h{h}"
    for h in hours
    for v in complete_vars
]

In [ ]:
to_stack = X_dyn[ordered]

# Which dtypes do we actually have?
print(to_stack.dtypes.value_counts())

# List the object-dtype (string) columns
bad = to_stack.dtypes[to_stack.dtypes == "object"].index.tolist()
print("non-numeric dynamic cols:", bad)

In [ ]:
# Filter to only true numeric columns
numeric_ordered = [c for c in ordered
                   if pd.api.types.is_numeric_dtype(X_dyn[c])]

# Re-compute 3-D array out of those:
arr = X_dyn[numeric_ordered].values.astype("float32")
N, HM = arr.shape
H     = len(hours)
M     = HM // H

X_3d = arr.reshape(N, H, M)
print("Now dynamic shape:", X_3d.shape, "dtype=", X_3d.dtype)

In [ ]:
static

In [ ]:
X_3d

# **TRAIN TEST SPLIT, IMPUTING AND NORMALIZATION**

In [ ]:
Xs = static.drop(columns=["subject_id","icustay_id","is_alive"]).values.astype("float32")
y = static["is_alive"].values

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(N)
tr_idx, te_idx = train_test_split(idx, test_size=0.2, stratify=y, random_state=42)

# Xs for static features, Xd for dynamic features

Xs_tr, Xs_te = Xs[tr_idx], Xs[te_idx]
Xd_tr, Xd_te = X_3d[tr_idx], X_3d[te_idx]
y_tr, y_te = y[tr_idx], y[te_idx]

In [ ]:
print("train:", Xs_tr.shape, Xd_tr.shape, y_tr.shape)
print("test: ", Xs_te.shape, Xd_te.shape, y_te.shape)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Static pipeline
imp_s = SimpleImputer(strategy="mean").fit(Xs_tr) # Imputer
sc_s = StandardScaler().fit(imp_s.transform(Xs_tr)) # StandardScaler

Xs_tr = sc_s.transform(imp_s.transform(Xs_tr))
Xs_te = sc_s.transform(imp_s.transform(Xs_te))

# Dynamic pipeline: reshape to (N, H*M), impute/scale, then back to (N,H,M)
Xd_tr_flat = Xd_tr.reshape(len(Xd_tr), -1)
Xd_te_flat = Xd_te.reshape(len(Xd_te), -1)

imp_d = SimpleImputer(strategy="mean").fit(Xd_tr_flat) # Imputer
sc_d = StandardScaler().fit(imp_d.transform(Xd_tr_flat))

Xd_tr = sc_d.transform(imp_d.transform(Xd_tr_flat)).reshape(-1, H, M)
Xd_te = sc_d.transform(imp_d.transform(Xd_te_flat)).reshape(-1, H, M)

# **Saving to NUMPY arrays**

In [ ]:
np.save("Xstatic_train_c1.npy", Xs_tr)
np.save("Xstatic_test_c1.npy", Xs_te)
np.save("Xdynamic_train_c1.npy", Xd_tr)
np.save("Xdynamic_test_c1.npy", Xd_te)
np.save("y_train_c1.npy", y_tr)
np.save("y_test_c1.npy",  y_te)

In [ ]:
static.to_csv("client_1_static.csv", index=False)

In [ ]:
client_1_dynamic = cohort[dyn_cols].copy()
client_1_dynamic.to_csv("client_1_dynamic.csv", index=False)

# **FOR FL for matching static features**

In [ ]:
# Xs_matching = static.drop(columns=["subject_id", "icustay_id", "admit_weekday", "is_weekend", "wday_sin", "wday_cos", "is_alive"]).copy()
Xs_local = static.drop(columns=["subject_id", "icustay_id", "is_alive", "age", "gender", "los_hours", "admit_hour", "hour_sin", "hour_cos"]).copy()

In [ ]:
# Summary of our csv file:
print(Xs_local.describe())

In [ ]:
# Summary of our csv file:
print(Xs_local.describe())

In [ ]:
non_numeric = [c for c in Xs_local.columns
               if not pd.api.types.is_numeric_dtype(Xs_local[c])]
if non_numeric:
    raise ValueError("Non-numeric column(s) in static:", non_numeric)

In [ ]:
Xs_local = Xs_local.values.astype("float32") 
y = static["is_alive"].values.astype("float32")

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(N)
tr_idx, te_idx = train_test_split(idx, test_size=0.2, stratify=y, random_state=42)


Xs_tr_local, Xs_te_local = Xs_local[tr_idx], Xs_local[te_idx]
y_tr_local, y_te_local = y[tr_idx], y[te_idx]

In [ ]:
print("train:", Xs_tr_local.shape)
print("test: ", Xs_te_local.shape)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Static pipeline
imp_s = SimpleImputer(strategy="mean").fit(Xs_tr_local) # Imputer
sc_s = StandardScaler().fit(imp_s.transform(Xs_tr_local)) # StandardScaler

Xs_tr_local = sc_s.transform(imp_s.transform(Xs_tr_local))
Xs_te = sc_s.transform(imp_s.transform(Xs_te_local))

In [ ]:
print("static NaNs in train:", np.isnan(Xs_tr_local).sum())
print("static NaNs in  test:", np.isnan(Xs_te_local).sum())

In [ ]:
np.save("Xstatic_train_local1.npy", Xs_tr_local)
np.save("Xstatic_test_local1.npy", Xs_te_local)

In [ ]:
np.save("y_train_local1.npy", y_tr_local)
np.save("y_test_mlocal1.npy",  y_te_local)